In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as spopt
import numpy.ma as ma
from tqdm import tqdm
import pandas as pd
import glob, os

import sys
#Need this command for the _merge function for recursion
sys.setrecursionlimit(10**5)

In [12]:
def master_opener():
    
    #Opening the most recent master_df, dont forget to pull the git repo
    master_df = pd.read_csv("/home/omar/git/logs/master_df.csv", sep=",", header=0, index_col=0, \
                            na_values='NA', engine='python')
    
    #Adding in the observed_hours per row
    master_df["observed_hours"] = (master_df["MJD_end_time"] - master_df["MJD_start_time"]) * 24
    
    #Sort just because 
    master_df = master_df.sort_values(by="Obs_Date_String")

    #R240121 & R240114
    #FRB20220912A & F220912
    df_r147 = master_df[(master_df["Source_Name"] == "R240121") | (master_df["Source_Name"] == "R240114") | \
            (master_df["Source_Name"] == "FRB20240114A") & (master_df["Telescope"] != "Nancay")]
    
    #Ef, Nt and Mc observed in pr247a/pr248a/pr249a   
    df_r147 = df_r147[(df_r147["Telescope"] == "wb") | (df_r147["Telescope"] == "tr") \
                        | (df_r147["Telescope"] == "stk") | (df_r147["Telescope"] == "o8")]

    df_r147 = df_r147[(df_r147["scan_name"] != "rk002")]
    
    display(df_r147)
    
    tels = set(df_r147["Telescope"].values)
    obs_set= set(df_r147["Central_freq_(MHz)"].values)
    print(len(obs_set))

    print(tels)
    print(obs_set)
    
    return df_r147

df_r117 = master_opener()

,scan_name,scan_no,Telescope,Source_Name,Obs_Date_String,MJD_start_time,RA_J2000,RA_J2000_(deg),Dec_J2000,Dec_J2000_(deg),...,bytes_in_file_header,MJD_end_time,complete_filename,Are_bytes_signed?,Polarization_order,Backend,Starting_subint,Subints_per_file,Source_Name_nancay,observed_hours
115514,pcn235,no0001,wb,R240121,2024-01-25T13:50:01,60334.576400,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,295.0,60334.586794,pcn235_wb_no0001_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249437
115515,pcn235,no0002,wb,R240121,2024-01-25T14:05:15,60334.586979,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,295.0,60334.597349,pcn235_wb_no0002_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248882
115516,pcn235,no0003,wb,R240121,2024-01-25T14:20:27,60334.597535,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,295.0,60334.607917,pcn235_wb_no0003_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164
115517,pcn235,no0004,wb,R240121,2024-01-25T14:35:39,60334.608090,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,295.0,60334.618472,pcn235_wb_no0004_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164
115518,pcn235,no0005,wb,R240121,2024-01-25T14:50:51,60334.618646,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,295.0,60334.629016,pcn235_wb_no0005_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129772,p47040,no0038,wb,R240114,2024-03-27T12:22:03,60396.515312,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,295.0,60396.525680,p47040_wb_no0038_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248818
129773,p47040,no0039,wb,R240114,2024-03-27T12:37:14,60396.525856,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,295.0,60396.536236,p47040_wb_no0039_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249110
129774,p47040,no0040,wb,R240114,2024-03-27T12:52:26,60396.536412,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,295.0,60396.546779,p47040_wb_no0040_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248818
129775,p47040,no0041,wb,R240114,2024-03-27T13:07:38,60396.546968,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,295.0,60396.557347,p47040_wb_no0041_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249110


7
{'wb', 'tr', 'stk', 'o8'}
{4708.0, 1381.46484375, 1414.0, 1418.0, 332.0, 1424.0, 1271.0}


In [13]:
#dictionary with frequency setups
def freq_dict_maker():
    
    #acces with: freq_dict[332]
    freq_dict = {
        332 : "P",
        1271 : "L18",
        1414 : "L15",
        1418 : "L12",      
        1381.46484375 : "L13",
        1424 : "L1",
        4708 : "C4"
        # 1250.15625 : "L19_1",
        # 1250.078125 : "L19_2",
        # 1350.15625 : "L20_2",
        # 1350.078125 : "L20_2",
        # 1300.15625 : "L21",
        # 410.0625 : "P_2"
    }
    
    return freq_dict

In [14]:
def _merge(head, tail):
    if tail == []:
        return head

    a, b = head[-1]
    x, y = tail[0]

    do_merge = b > x
    if do_merge:
        head_ = head[:-1] + [(a, max(b, y))]
        tail_ = tail[1:]
        return _merge(head_, tail_)
    else:
        head_ = head + tail[:1]
        tail_ = tail[1:]
        return _merge(head_, tail_)


def merge_intervals(lst):
    if len(lst) <= 1:
        return lst
    
    #sort on the first element of the tuples for the whole list
    lst = sorted(lst, key=lambda x: x[0])

    return _merge(lst[:1], lst[1:])

def get_overlap_times(df, coln1="MJD_start", coln2="MJD_end"):
    
    # get the right columns
    start = np.array(df["MJD_start_time"])
    end = np.array(df["MJD_end_time"])
    
    # check if input is correct:
    for s, e in zip(start, end):
        if s >= e:
            print("WARNING: start time should not exceed end time")
            print(f"{start} and {end}")
            
    # create tuple pairs:
    times = [(x, y) for x, y in zip(start, end)]
    times_reduced = merge_intervals(times)
    
    return times_reduced

def calc_overlap_times(times, unit="days"):
    """ Times is a list of tuple pairs, e.g.:
    [(1, 3), (5.5, 6.0)]
    Will return the sum of the differences of the pairs:
    [(1, 3), (5.5, 6.0)] --> (3-1) + (6.0-5.5) = 2 + 0.5 = 2.5 """
    ttotal = 0 # total time
    for timepair in times:
        s, e = timepair[0], timepair[1]
        ttotal += (e - s)
    print(f"The total non overlapping observing time is {ttotal:.2f} {unit} or {(ttotal*24):.2f} hours")

In [16]:
def hours_calc(df_source, total_time=False, activity_phase=False, save_bool=False):
    
    #call the freq dictionary
    freq_dict = freq_dict_maker()
    
    #Conmbine all the pickles files into one big dataframe    
    df_source["total_obs_time_s"] = (df_source["MJD_end_time"] - df_source["MJD_start_time"]) * 24 * 3600
        
    #print setup of observing frequenies
    freqs = df_source['Central_freq_(MHz)'].values
    print(set(freqs))
    
    ##If you want to limit in time:
    if activity_phase == True:        
        
        #### Main R117 campaign -- Observational setup ####
        #59866 - 14-Oct-22 00:00:00
        #59983 - 08-Feb-23 00:00:00
#         bound_l = 59866 
#         bound_r = 59983         
        
        #### Overlap with FAST -- Comp plot #####
        #59910-59869 -- David code
        #59602 start of wb; 59644 last Stk obs
        #59910
#         bound_l = 59869 
#         bound_r = 59910           
        
        #### Activity phase with ATA #####
        ###Our campaign: 59867 -- 59983
        ###ATA campaign: 59872 -- 59933
        #Begin of Europe observavtions: 59867
        #End of ATA observations: 59933
        bound_l = 59867
        bound_r = 59933
        
        print(f"!!Using Activity phase!! begin: {bound_l} | end: {bound_r}")
        df_source = df_source[(df_source['MJD_start_time'] > bound_l) & (df_source['MJD_end_time'] < bound_r)]
    
    #Westerbork
    df_wb = df_source[df_source["Telescope"] == 'wb']
    wb_obs_h = sum(df_wb["total_obs_time_s"].values)/3600
    
    #Torun
    df_tr = df_source[df_source["Telescope"] == 'tr']
    tr_obs_h = sum(df_tr["total_obs_time_s"].values)/3600
    
    #Stockert
    df_stk = df_source[df_source["Telescope"] == 'stk']
    stk_obs_h = sum(df_stk["total_obs_time_s"].values)/3600
    
    #Onsala
    df_o8 = df_source[df_source["Telescope"] == 'o8']
    o8_obs_h = sum(df_o8["total_obs_time_s"].values)/3600

    # #Dwingeloo
    # df_dwi = df_source[df_source["Telescope"] == 'dwi']
    # dwi_obs_h = sum(df_dwi["total_obs_time_s"].values)/3600
    
    #Print observing hours
    print(f"total obs Wb {wb_obs_h:.2f} h")
    print(f"total obs Tr {tr_obs_h:.2f} h")
    print(f"total obs Stk {stk_obs_h:.2f} h")
    print(f"total obs o8 {o8_obs_h:.2f} h")
    #print(f"total obs Dwi {dwi_obs_h:.2f} h")
    
    total_obs = sum(df_source["total_obs_time_s"].values) / 3600
    print(f"total observing time: {total_obs:.2f} h")
    print("--------------------")
    
    print("-P band-")
    #Pband
    df_r117_pband = df_source[(df_source["Central_freq_(MHz)"] == 332.0)]
    times_p = get_overlap_times(df_r117_pband)
    ttotal_p = calc_overlap_times(times_p)
    
    print("-C band-")
    df_r117_cband = df_source[(df_source["Central_freq_(MHz)"] == 4708.0)]
    
    times_c = get_overlap_times(df_r117_cband)
    ttotal_c = calc_overlap_times(times_c)
    
    print("-L band-")
    #Lband
    df_r117_lband = df_source[(df_source["Central_freq_(MHz)"] != 332.0) & \
                          (df_source["Central_freq_(MHz)"] != 4708.0) ]
    times_l = get_overlap_times(df_r117_lband)
    ttotal_l = calc_overlap_times(times_l)
    
    #Calculate the unique hours in the observations
    if total_time == True:
        print("--------------------")
        times = get_overlap_times(df_source)
        ttotal = calc_overlap_times(times)
        print("--------------------")
    
    #Counters for observing time
    total_h_obs = 0
    total_h_obs_lband = 0
    
    #Loop over all entries in the freq_dict
    for obser_freq in freq_dict:

        #Create mask per frequency setup
        df_freq_setup_mask = (df_source['Central_freq_(MHz)'] == obser_freq)
        df_freq_setup = df_source[(df_freq_setup_mask)]
        
        #As a double check; get the set of all telescopes in this setup (should be unique)
        part_telescope = set(df_freq_setup["Telescope"])
        
        #Amount of hours per frequency setup in hours
        df_freq_setup_h = sum(df_freq_setup["total_obs_time_s"].values)/3600
        print(f"{freq_dict[obser_freq]}: time = {df_freq_setup_h:.2f} h -- central_freq {obser_freq} -- {part_telescope}")
        
        #Add to total
        total_h_obs += df_freq_setup_h

        #Add if observation is on Lband
        if freq_dict[obser_freq].startswith("L"):
            total_h_obs_lband += df_freq_setup_h
    
    print(f"Total observing on L-band: {total_h_obs_lband:.2f}")
    print(f"as a double check, the total is over the freq setup is: {total_h_obs:.2f}")
    print("--------------------")
    
    ##Saving the full campaign (activity_phase=True) 59866-59983
    ##Ran on 02-08-2023
    if save_bool == True:
        display(df_source)
        df_source = df_source.sort_values(by=['MJD_start_time'])
        csv_obs = "../dbs/r147_fullcampaign.csv"
        df_source.to_csv(csv_obs, sep=",", header=True, index=True, na_rep='NA')
        print("campaign saved")
        
    return df_source

df_r117_campaign = hours_calc(df_r117, total_time=True, activity_phase=False, save_bool=True)

{4708.0, 1381.46484375, 1414.0, 1418.0, 332.0, 1424.0, 1271.0}
total obs Wb 409.05 h
total obs Tr 71.98 h
total obs Stk 473.51 h
total obs o8 109.52 h
total observing time: 1064.07 h
--------------------
-P band-
The total non overlapping observing time is 12.59 days or 302.16 hours
-C band-
The total non overlapping observing time is 0.89 days or 21.36 hours
-L band-
The total non overlapping observing time is 22.40 days or 537.68 hours
--------------------
The total non overlapping observing time is 23.90 days or 573.61 hours
--------------------
P: time = 302.16 h -- central_freq 332 -- {'wb'}
L18: time = 106.90 h -- central_freq 1271 -- {'wb'}
L15: time = 46.67 h -- central_freq 1414 -- {'tr'}
L12: time = 3.96 h -- central_freq 1418 -- {'tr'}
L13: time = 473.51 h -- central_freq 1381.46484375 -- {'stk'}
L1: time = 109.52 h -- central_freq 1424 -- {'o8'}
C4: time = 21.36 h -- central_freq 4708 -- {'tr'}
Total observing on L-band: 740.55
as a double check, the total is over the freq 

,scan_name,scan_no,Telescope,Source_Name,Obs_Date_String,MJD_start_time,RA_J2000,RA_J2000_(deg),Dec_J2000,Dec_J2000_(deg),...,MJD_end_time,complete_filename,Are_bytes_signed?,Polarization_order,Backend,Starting_subint,Subints_per_file,Source_Name_nancay,observed_hours,total_obs_time_s
115514,pcn235,no0001,wb,R240121,2024-01-25T13:50:01,60334.576400,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,60334.586794,pcn235_wb_no0001_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249437,897.974337
115515,pcn235,no0002,wb,R240121,2024-01-25T14:05:15,60334.586979,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,60334.597349,pcn235_wb_no0002_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248882,895.975488
115516,pcn235,no0003,wb,R240121,2024-01-25T14:20:27,60334.597535,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,60334.607917,pcn235_wb_no0003_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164,896.991296
115517,pcn235,no0004,wb,R240121,2024-01-25T14:35:39,60334.608090,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,60334.618472,pcn235_wb_no0004_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164,896.991296
115518,pcn235,no0005,wb,R240121,2024-01-25T14:50:51,60334.618646,21:34:03.4440,323.514350,04:29:06.1800,4.485050,...,60334.629016,pcn235_wb_no0005_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248882,895.975488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129772,p47040,no0038,wb,R240114,2024-03-27T12:22:03,60396.515312,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,60396.525680,p47040_wb_no0038_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248818,895.746176
129773,p47040,no0039,wb,R240114,2024-03-27T12:37:14,60396.525856,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,60396.536236,p47040_wb_no0039_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249110,896.794752
129774,p47040,no0040,wb,R240114,2024-03-27T12:52:26,60396.536412,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,60396.546779,p47040_wb_no0040_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.248818,895.746176
129775,p47040,no0041,wb,R240114,2024-03-27T13:07:38,60396.546968,21:27:39.8354,321.915981,04:19:45.6344,4.329343,...,60396.557347,p47040_wb_no0041_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249110,896.794753


campaign saved


In [30]:
#df_r117_campaign = hours_calc(df_r117, total_time=True, activity_phase=False, save_bool=False)